# nb04 - Delta Lake: MERGE, schema evolution, time travel, CDF, OPTIMIZE

So far silver was `overwrite`-everything. Real ingestion is **incremental**: new batches
arrive with new rows, changed rows, and occasionally new columns. Delta turns the table
directory into an ACID log (`_delta_log/`), which is what makes all of that safe.

**You will learn:** idempotent upserts with `MERGE`, `mergeSchema` evolution, time travel
(`DESCRIBE HISTORY` / `AS OF`), the change data feed (CDF), and file compaction with
`OPTIMIZE` + data skipping.

**Prerequisite:** nb02 (silver tables). Everything here is re-runnable - the MERGE cell
simulates "batch 2" with updates and a new column, and is written to be idempotent.


In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

from de_stack import get_spark, BRONZE, register_table

spark = get_spark("nb04-delta", shuffle_partitions=8)

for db, name in [("silver", "orders"), ("silver", "customers")]:
    if not register_table(spark, db, name):
        raise RuntimeError(f"silver.{name} missing - run nb02 first")
print("ready")


12:04:51.208 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


ready


12:04:57.923 WARN CreateNamespaceExec: Namespace silver was created concurrently. Ignoring.


## 1. What's in the log?

The `_delta_log` directory is a chronological journal: one JSON commit per version,
each with the full action set (add/remove file, metadata, protocol). `DESCRIBE HISTORY`
reads it back.


In [2]:
import os

log_dir = "/data/warehouse/silver.db/orders/_delta_log"
print(sorted(os.listdir(log_dir))[:5], "...")
spark.sql("DESCRIBE HISTORY silver.orders").select("version", "operation", "operationMetrics.numFiles", "timestamp").show(truncate=False)


['.00000000000000000000.crc.crc', '.00000000000000000000.json.crc', '.00000000000000000001.crc.crc', '.00000000000000000001.json.crc', '.00000000000000000002.crc.crc'] ...


+-------+---------------------------------+--------+-----------------------+
|version|operation                        |numFiles|timestamp              |
+-------+---------------------------------+--------+-----------------------+
|5      |OPTIMIZE                         |NULL    |2026-10-13 11:38:53.82 |
|4      |SET TBLPROPERTIES                |NULL    |2026-10-13 11:38:31.76 |
|3      |MERGE                            |NULL    |2026-10-13 11:38:15.644|
|2      |MERGE                            |NULL    |2026-10-13 11:37:52.342|
|1      |MERGE                            |NULL    |2026-10-13 11:37:37.613|
|0      |CREATE OR REPLACE TABLE AS SELECT|124     |2026-10-13 11:34:59.144|
+-------+---------------------------------+--------+-----------------------+



## 2. MERGE: the idempotent upsert

Build a "next batch" from bronze: same `order_id`s, some with **changed** amounts
(supplier price correction), plus new orders. `MERGE ... WHEN MATCHED THEN UPDATE
WHEN NOT MATCHED THEN INSERT` applies it safely - re-running the same cell is a no-op
for already-applied changes (idempotency you can verify below).


In [3]:
raw = spark.read.json(f"{BRONZE}/orders_raw")
batch2 = (
    raw.limit(3000)
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("quantity", F.expr("try_cast(quantity AS INT)"))
    .withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price") * 1.07, 2))  # +7% correction
    .withColumn("order_date", F.to_date("order_ts"))
    .select("order_id", "order_ts", "customer_id", "product_id", "quantity",
            "unit_price", "total_amount", "currency", "channel", "status", "order_date")
    .dropDuplicates(["order_id"])
)

# pretend we already have a couple of the new ids so INSERT branch also fires
new_rows = batch2.limit(50).withColumn("order_id", F.concat(F.lit("O-NEW-"), F.monotonically_increasing_id()))
batch2 = batch2.unionByName(new_rows)
batch2.show(3, truncate=False)
print("batch2 rows:", batch2.count())


+--------+-------------------+-----------+----------+--------+----------+------------+--------+-------+---------+----------+
|order_id|order_ts           |customer_id|product_id|quantity|unit_price|total_amount|currency|channel|status   |order_date|
+--------+-------------------+-----------+----------+--------+----------+------------+--------+-------+---------+----------+
|O900051 |2026-09-27 11:14:32|C104648    |P10404    |5       |13.77     |73.67       |EUR     |web    |shipped  |2026-09-27|
|O900069 |2026-10-03 21:06:05|C107466    |P10082    |1       |21.25     |22.74       |USD     |web    |created  |2026-10-03|
|O900104 |2026-10-10 15:50:31|C103521    |P10252    |1       |27.53     |29.46       |EUR     |ios_app|delivered|2026-10-10|
+--------+-------------------+-----------+----------+--------+----------+------------+--------+-------+---------+----------+
only showing top 3 rows


batch2 rows: 3043


In [4]:
# explicit column maps: batch2 only carries the ingestion payload; columns it does not
# know (customer_note, items, ...) stay untouched on matched rows
payload = {c: f"s.{c}" for c in batch2.columns}

# condition on the payload: once a row matches the batch (same amount), it stops updating
DeltaTable.forName(spark, "silver.orders").alias("t").merge(
    batch2.alias("s"),
    "t.order_id = s.order_id",
).whenMatchedUpdate(set=payload, condition="s.total_amount <> t.total_amount")  .whenNotMatchedInsert(values=payload)  .execute()

spark.sql("DESCRIBE HISTORY silver.orders").select("version", "operation", "operationMetrics.numTargetRowsUpdated", "operationMetrics.numTargetRowsInserted").show(truncate=False)


12:05:10.489 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


12:05:31.730 WARN MapPartitionsRDD: RDD 52 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+-------+---------------------------------+--------------------+---------------------+
|version|operation                        |numTargetRowsUpdated|numTargetRowsInserted|
+-------+---------------------------------+--------------------+---------------------+
|6      |MERGE                            |0                   |0                    |
|5      |OPTIMIZE                         |NULL                |NULL                 |
|4      |SET TBLPROPERTIES                |NULL                |NULL                 |
|3      |MERGE                            |58797               |0                    |
|2      |MERGE                            |0                   |0                    |
|1      |MERGE                            |2993                |50                   |
|0      |CREATE OR REPLACE TABLE AS SELECT|NULL                |NULL                 |
+-------+---------------------------------+--------------------+---------------------+



In [5]:
# idempotency check: re-applying the same batch must update nothing
spark.sql("DESCRIBE HISTORY silver.orders").select("version").first()[0]

DeltaTable.forName(spark, "silver.orders").alias("t").merge(
    batch2.alias("s"), "t.order_id = s.order_id",
).whenMatchedUpdate(set=payload, condition="s.total_amount <> t.total_amount")  .whenNotMatchedInsert(values=payload)  .execute()

hist = spark.sql("DESCRIBE HISTORY silver.orders")
last = hist.orderBy(F.col("version").desc()).first()
print("second MERGE ->", last["operation"], "| rows updated:", last["operationMetrics"].get("numTargetRowsUpdated"))


12:05:50.558 WARN MapPartitionsRDD: RDD 141 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


second MERGE -> MERGE | rows updated: 0


## 3. Schema evolution

Bronze orders carry `risk_flags` (an array). A later enrichment step wants to publish a
derived `risk_flag_count` column the silver table never had. Delta refuses to silently
add columns - you have to opt in, either per-write (`mergeSchema`) or per-session
(`spark.databricks.delta.schema.autoMerge.enabled`).


In [6]:
evolved = (
    spark.read.json(f"{BRONZE}/orders_raw")
    .select("order_id", F.size("risk_flags").alias("risk_flag_count"))
    .dropDuplicates(["order_id"])
)

def enrich_risk_counts():
    return (DeltaTable.forName(spark, "silver.orders").alias("t")
            .merge(evolved.alias("s"), "t.order_id = s.order_id")
            .whenMatchedUpdate(set={"risk_flag_count": "s.risk_flag_count"})
            .whenNotMatchedInsert(values={"order_id": "s.order_id",
                                          "risk_flag_count": "s.risk_flag_count"}))

try:
    enrich_risk_counts().execute()
except Exception as e:
    print("refused as expected:", str(e).splitlines()[0][:120])

# opt in: session-level schema auto-merge for MERGE operations
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
enrich_risk_counts().execute()
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")

spark.sql("DESCRIBE TABLE silver.orders").show(40, truncate=False)


12:06:11.002 WARN MapPartitionsRDD: RDD 218 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


12:06:25.298 WARN MapPartitionsRDD: RDD 280 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+-----------------------+-------------------------------------------------+-------+
|col_name               |data_type                                        |comment|
+-----------------------+-------------------------------------------------+-------+
|channel                |string                                           |NULL   |
|currency               |string                                           |NULL   |
|customer_id            |string                                           |NULL   |
|customer_note          |string                                           |NULL   |
|items                  |array<struct<price:double,qty:bigint,sku:string>>|NULL   |
|loyalty_tier           |string                                           |NULL   |
|order_id               |string                                           |NULL   |
|order_ts               |timestamp                                        |NULL   |
|payment_method         |string                                           |N

## 4. Time travel

Every read can be pinned to a **version** or a **timestamp** - audits, reproducible
training sets, or "roll back the bad batch without losing it".


In [7]:
current = spark.sql("SELECT COUNT(*) AS n, ROUND(SUM(total_amount), 2) AS revenue FROM silver.orders").first()
v0 = spark.sql("DESCRIBE HISTORY silver.orders").orderBy("version").first()["version"]
past = spark.sql(f"SELECT COUNT(*) AS n, ROUND(SUM(total_amount), 2) AS revenue FROM silver.orders VERSION AS OF {v0}").first()

print(f"now      v-latest: {current.n} rows, revenue {current.revenue}")
print(f"history v{v0}: {past.n} rows, revenue {past.revenue}")


now      v-latest: 58847 rows, revenue 4268144.43
history v0: 58797 rows, revenue 4248555.24


## 5. Change data feed (CDF)

Delta records row-level changes between versions. CDF powers CDC consumers and
incremental downstream jobs - nb06 revisits this with Spark 4.2's new `CHANGES` SQL
clause on top.


In [8]:
# CDF must be recorded from the table's first version - nb02 enables it at creation;
# the ALTER is an idempotent safety net for tables created before that setting.
# (A table created WITHOUT cdf only records changes from the ALTER version onward -
#  if yours predates it, rebuild silver via nb02 and its whole history is covered.)
spark.sql("ALTER TABLE silver.orders SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

cdf = (spark.read.format("delta")
       .option("readChangeFeed", "true")
       .option("startingVersion", 1)
       .table("silver.orders"))

cdf.select("_change_type", "order_id", "total_amount").show(5, truncate=False)
print("change types:", [r[0] for r in cdf.select("_change_type").distinct().collect()])


+----------------+--------+------------+
|_change_type    |order_id|total_amount|
+----------------+--------+------------+
|update_preimage |O900135 |81.74       |
|update_postimage|O900135 |81.74       |
|update_preimage |O900519 |54.21       |
|update_postimage|O900519 |54.21       |
|update_preimage |O900569 |174.06      |
+----------------+--------+------------+
only showing top 5 rows


change types: ['update_postimage', 'update_preimage', 'insert']


## 6. OPTIMIZE: fixing nb03's small files

Remember the 6 tiny files per partition from nb02? `OPTIMIZE` rewrites files per
partition up to a target size (default 128 MB). Combined with **Z-order** on a column
you filter by, Delta can also *skip* data files at read time using file statistics.


In [9]:
import os

partitions = sorted(d for d in os.listdir("/data/warehouse/silver.db/orders") if d.startswith("order_date="))
sample = f"/data/warehouse/silver.db/orders/{partitions[0]}"
n_before = len([f for f in os.listdir(sample) if f.startswith("part-")])

spark.sql("OPTIMIZE silver.orders").select("metrics").first()[0]

n_after = len([f for f in os.listdir(sample) if f.startswith("part-")])
print(f"small-files compaction in {partitions[0]}: {n_before} -> {n_after} data files")


small-files compaction in order_date=2026-09-13: 12 -> 12 data files


In [10]:
spark.sql("OPTIMIZE silver.orders ZORDER BY (customer_id)")
print("zorder done - now compare reads with and without the customer_id predicate:")
spark.sql("SELECT COUNT(*) FROM silver.orders WHERE customer_id = 'C104346'").show()


zorder done - now compare reads with and without the customer_id predicate:


+--------+
|count(1)|
+--------+
|       3|
+--------+



## Wrap-up

You can now evolve a table safely: upsert, add columns, look back in time, feed
increments downstream, and compact. Next: **nb05_structured_streaming.ipynb** - the same
Delta tables, but fed continuously from the event generator.
